# Demo 08 : Dynamic subagents

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · fan-out-in-code

We first run the application-owned Python loop. Then a Deep Agent writes its own JavaScript fan-out in eval. The script calls the configured manuals subagent once per error code through task and collects both answers. The configured team stays fixed; the script chooses the dispatches.

In [2]:
import json
import sqlite3
import threading
import time
from pathlib import Path

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langchain_core.callbacks import BaseCallbackHandler
from deepagents import create_deep_agent
from deepagents.backends import StateBackend, FilesystemBackend, StoreBackend, CompositeBackend
from deepagents.backends.utils import create_file_data
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


class Counter(BaseCallbackHandler):
    def __init__(self):
        self.lock = threading.Lock()
        self.reset()

    def reset(self):
        self.calls = self.input = self.output = self.invalid = self.errors = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        with self.lock:
            self.calls += 1

    def on_llm_end(self, response, **kwargs):
        with self.lock:
            for generations in response.generations:
                for generation in generations:
                    message = getattr(generation, "message", None)
                    usage = getattr(message, "usage_metadata", None) or {}
                    self.input += usage.get("input_tokens", 0)
                    self.output += usage.get("output_tokens", 0)
                    self.invalid += len(getattr(message, "invalid_tool_calls", []) or [])

    def on_llm_error(self, error, **kwargs):
        with self.lock:
            self.errors += 1


counter = Counter()
llm = make_llm(callbacks=[counter], max_tokens=1800, max_retries=0)
metrics = []
LIMIT = {"recursion_limit": 24}
DB = Path(data("demo-world/service.db")).resolve()
MANUALS = Path(data_folder("demo-world/manuals"))
PAGES = {p.name: p.read_text(encoding="utf-8") for p in sorted(MANUALS.glob("*.md"))}
WORKSPACE = (Path.cwd() / "workspace").resolve()
WORKSPACE.mkdir(exist_ok=True)


def ask(text):
    return {"messages": [HumanMessage(text)]}


def answer(result):
    return str(next(m.content for m in reversed(result["messages"])
                    if isinstance(m, AIMessage) and m.content and not m.tool_calls))


def measured(label, operation):
    counter.reset()
    start = time.perf_counter()
    try:
        return operation()
    finally:
        row = dict(label=label, calls=counter.calls, input_tokens=counter.input,
                   output_tokens=counter.output, malformed=counter.invalid, errors=counter.errors,
                   seconds=round(time.perf_counter() - start, 1))
        metrics.append(row)
        print("metrics :", json.dumps(row))


async def measured_async(label, operation):
    counter.reset()
    start = time.perf_counter()
    try:
        return await operation()
    finally:
        row = dict(label=label, calls=counter.calls, input_tokens=counter.input,
                   output_tokens=counter.output, malformed=counter.invalid, errors=counter.errors,
                   seconds=round(time.perf_counter() - start, 1))
        metrics.append(row)
        print("metrics :", json.dumps(row))


def show(result):
    print("tools :", [m.name for m in result["messages"] if m.type == "tool"])
    print("answer :", answer(result))


@tool
def part_stock(part: str) -> str:
    """Look up stock, storage location and name for a Veenhaven Bus part number."""
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        rows = db.execute("SELECT part_no, name, stock, depot FROM parts WHERE part_no = ?",
                          (part.upper(),)).fetchall()
    return json.dumps(rows)


@tool
def fault_guidance(error_code: str) -> str:
    """Find the manual's meaning, first action and replacement condition for a bus error code."""
    return "\n".join(f"[{name}]\n{line}" for name, text in PAGES.items()
                     for line in text.splitlines()
                     if line.startswith("| " + error_code.upper() + " |")) or "Unknown error code."


STOCK_TASK = "Look up CE-CF-02 stock and location. Answer briefly from the tool."
MANUAL_TASK = "Look up F-CE-110. Give the first action and replacement condition, citing the file."
QUESTION = ("Bus CE12-2021-0117 shows F-CE-110. How many CE-CF-02 cooling fans are in stock, "
            "at which location, and what are the manual's first action and replacement condition? Be brief.")

In [3]:
worker = create_agent(llm, tools=[fault_guidance],
    system_prompt="Use fault_guidance. Return the first action in one sentence.")
codes = ["F-CE-110", "F-CE-305"]

def fanout():
    found = []
    for code in codes:
        found.append(answer(worker.invoke(ask("First action for " + code + "."), LIMIT)))
    return found

print("Python answers :", measured("python-fanout", fanout))

metrics : {"label": "python-fanout", "calls": 4, "input_tokens": 465, "output_tokens": 69, "malformed": 0, "errors": 0, "seconds": 12.1}
Python answers : ['Check the traction cooling fan.', 'Inspect the door actuator.']


In [4]:
import xml.etree.ElementTree as ET
from langchain_quickjs import CodeInterpreterMiddleware

def show_eval(result):
    values = []
    for message in result["messages"]:
        for call in getattr(message, "tool_calls", []):
            if call["name"] == "eval":
                print("JavaScript :\n" + call["args"]["code"])
        if message.type == "tool" and message.name == "eval":
            print("sandbox :", message.content)
            values.append(json.loads(ET.fromstring(message.content).text))
    show(result)
    assert values, "The agent must run interpreter code."
    return values[-1]

manuals = {"name": "manuals", "description": "Finds the first action for a bus error code.",
           "tools": [fault_guidance],
           "system_prompt": "Use fault_guidance once. Return the first action briefly. Do not use files."}
dispatcher = create_deep_agent(model=llm, subagents=[manuals], middleware=[
    CodeInterpreterMiddleware(mode="turn", timeout=2, max_ptc_calls=2,
                              max_subagent_concurrency=2, max_result_chars=1600)],
    system_prompt=("Use only eval, with one JavaScript script. Use Promise.all and map to call "
        "top-level task once per supplied error code, subagentType manuals. "
        "Give each dispatch responseSchema for an object with error_code and action, both strings. "
        "Return JSON.stringify of the collected objects as the last expression. Then answer briefly."))
result = await measured_async("quickjs-fanout", lambda: dispatcher.ainvoke(ask(
    "Run a first-action lookup for F-CE-110 and F-CE-305 using interpreter fan-out. "
    "Each task description must include its error code."), LIMIT))
found = show_eval(result)
print("coverage :", sorted(row["error_code"] for row in found))
assert sorted(row["error_code"] for row in found) == sorted(codes)
assert all(row["action"] for row in found)

metrics : {"label": "quickjs-fanout", "calls": 6, "input_tokens": 16595, "output_tokens": 310, "malformed": 0, "errors": 0, "seconds": 16.2}
JavaScript :
const codes = ["F-CE-110", "F-CE-305"];
const results = await Promise.all(codes.map((error_code) => task({
  description: `Find the first action for bus error code ${error_code}. Return the error_code and first action.`,
  subagentType: "manuals",
  responseSchema: {
    type: "object",
    properties: {
      error_code: { type: "string" },
      action: { type: "string" }
    },
    required: ["error_code", "action"]
  }
})));
JSON.stringify(results);
sandbox : <result>[{"error_code":"F-CE-110","action":"Check the traction cooling fan."},{"error_code":"F-CE-305","action":"Inspect and replace the door actuator."}]</result>
tools : ['eval']
answer : [{"error_code":"F-CE-110","action":"Check the traction cooling fan."},{"error_code":"F-CE-305","action":"Inspect and replace the door actuator."}]
coverage : ['F-CE-110', 'F-CE-305']


The JavaScript API uses `subagentType`, while the normal Python task tool uses `subagent_type`. `responseSchema` gives the script structured objects and is supported by declarative subagents. The concurrency cap limits simultaneous dispatches per REPL. It does not cap the total number of tasks, and `max_ptc_calls` counts `tools.*` calls only. A task inside eval skips the parent's per-task approval; we can gate eval or put approval inside the worker.

## b · splitting-a-long-input

The manuals stay in a Python variable. One call receives every manual. The other path splits the files in code and calls the model on each third, so no call holds the whole set. We then let the Deep Agent load these chunks into JavaScript and dispatch the same model on each piece. The root sees compact findings while the source text stays in the environment.

In [5]:
from langchain_core.messages.utils import count_tokens_approximately

names = sorted(PAGES)
joined = "\n\n".join(f"[{name}]\n{PAGES[name]}" for name in names)
chunks = []
step = (len(names) + 2) // 3
for start in range(0, len(names), step):
    group = names[start:start + step]
    chunks.append("\n\n".join(f"[{name}]\n{PAGES[name]}" for name in group))
print("files :", len(names), "characters :", len(joined))
print("approximate tokens :", count_tokens_approximately([HumanMessage(joined)]),
      [count_tokens_approximately([HumanMessage(chunk)]) for chunk in chunks])

def flat(message):
    content = message.content
    if isinstance(content, list):
        return "".join(block.get("text", "") if isinstance(block, dict) else str(block) for block in content)
    return str(content)

def whole():
    return flat(llm.invoke("Manuals follow. First action for F-CE-110, one sentence.\n\n" + joined))

def split_calls():
    sizes = []
    found = []
    for index, chunk in enumerate(chunks, start=1):
        before = counter.input
        text = flat(llm.invoke(
            "Excerpt of bus manuals. If F-CE-110 appears, give its first action in one sentence. "
            "Otherwise reply NONE.\n\n" + chunk))
        sizes.append(counter.input - before)
        found.append(text)
        print("chunk", index, ":", text[:220])
    print("split input tokens :", sizes, "max :", max(sizes))
    return [text for text in found if "NONE" not in text.upper()]

print("whole :", measured("whole-prompt", whole))
print("split :", measured("split-calls", split_calls))

files : 31 characters : 19626
approximate tokens : 4911 [1724, 1924, 1271]


metrics : {"label": "whole-prompt", "calls": 1, "input_tokens": 5121, "output_tokens": 24, "malformed": 0, "errors": 0, "seconds": 2.1}
whole : Check the CE-12 traction cooling fan airflow; replace it if airflow is below 80 percent.


chunk 1 : Check the traction cooling fan; replace it if airflow is below 80 percent.


chunk 2 : Check the traction cooling fan; replace it if airflow is below 80 percent.


chunk 3 : NONE
split input tokens : [1784, 2010, 1398] max : 2010
metrics : {"label": "split-calls", "calls": 3, "input_tokens": 5192, "output_tokens": 45, "malformed": 0, "errors": 0, "seconds": 7.2}
split : ['Check the traction cooling fan; replace it if airflow is below 80 percent.', 'Check the traction cooling fan; replace it if airflow is below 80 percent.']


In [6]:
from langchain.agents.middleware import wrap_model_call

usage_rows = []
root_has_corpus = []

class UsageRows(BaseCallbackHandler):
    def __init__(self, role):
        self.role = role

    def on_llm_end(self, response, **kwargs):
        for generations in response.generations:
            for generation in generations:
                usage = getattr(generation.message, "usage_metadata", None) or {}
                usage_rows.append({"role": self.role, "input": usage.get("input_tokens", 0),
                                   "output": usage.get("output_tokens", 0)})

@wrap_model_call
async def inspect_root(request, handler):
    messages = ([request.system_message] if request.system_message else []) + list(request.messages)
    text = "\n".join(flat(message) for message in messages)
    root_has_corpus.append(any(chunk in text for chunk in chunks))
    return await handler(request)

@tool
def manual_chunks() -> str:
    """Load JSON containing three bus-manual text chunks. Use them inside the interpreter."""
    return json.dumps(chunks)

root_llm = make_llm(callbacks=[counter, UsageRows("root")], max_tokens=1800, max_retries=0)
reader_llm = make_llm(callbacks=[counter, UsageRows("reader")], max_tokens=1800, max_retries=0)
reader = create_agent(reader_llm, system_prompt=(
    "Read this excerpt only. If F-CE-110 appears, give its first action and replacement condition "
    "in one sentence. Otherwise reply NONE."))
environment_agent = create_deep_agent(model=root_llm, subagents=[
    {"name": "reader", "description": "Reads one derived manual chunk.", "runnable": reader}],
    middleware=[CodeInterpreterMiddleware(ptc=[manual_chunks], mode="turn", timeout=2,
                max_ptc_calls=1, max_subagent_concurrency=1, max_result_chars=1600), inspect_root],
    system_prompt=("Use only eval once. Load JSON.parse(await tools.manualChunks({})) into a variable. "
        "For every chunk call top-level task with subagentType reader and description containing "
        "that chunk plus the F-CE-110 question. This is a compiled reader: do not use responseSchema. "
        "Collect findings with Promise.all and map. Return JSON.stringify of findings only, "
        "never the chunks. Then give a brief answer."))
result = await measured_async("quickjs-environment", lambda: environment_agent.ainvoke(ask(
    "Find the first action and replacement condition for F-CE-110 across the manual chunks. "
    "Keep the bulk input in the interpreter and delegate one lookup per chunk."), LIMIT))
findings = show_eval(result)
print("provider usage :", json.dumps(usage_rows))
print("root received a complete chunk :", root_has_corpus)
print("largest root input :", max(row["input"] for row in usage_rows if row["role"] == "root"))
print("largest reader input :", max(row["input"] for row in usage_rows if row["role"] == "reader"))
assert len(findings) == len(chunks)
assert not any(root_has_corpus)
assert any("80" in text and "fan" in text.lower() for text in findings)

metrics : {"label": "quickjs-environment", "calls": 5, "input_tokens": 15635, "output_tokens": 212, "malformed": 0, "errors": 0, "seconds": 13.2}
JavaScript :
const chunks = JSON.parse(await tools.manualChunks({})); const findings = await Promise.all(chunks.map((chunk) => task({ subagentType: "reader", description: `Inspect this manual chunk for the question: Find the first action and replacement condition for F-CE-110. Return only the relevant finding, quoting the action and replacement condition and noting any ambiguity. Chunk: ${typeof chunk === "string" ? chunk : JSON.stringify(chunk)}` }))); JSON.stringify(findings);
sandbox : <result>["First action: “check the traction cooling fan”; replacement condition: “replace it if airflow is below 80 percent.” No ambiguity.","First action: “check the traction cooling fan”; replacement condition: “replace it if airflow is below 80 percent.” No ambiguity.","NONE"]</result>
tools : ['eval']
answer : The first action is to **check the traction 

We use a compiled reader with the same model and no extra tools. The root writes the orchestration code; the loader puts the input in its environment without a parent ToolMessage containing that input. This is one level of the RLM idea. These manuals fit in a single prompt, so extra harness tokens and calls can outweigh the smaller reader prompts. A reader can miss a relevant passage and return NONE. We compare each printed finding with its source chunk.

## c · delegation-policy

The desk agent has both bus tools and two specialists, so it can answer itself or delegate. We run the same question in four arms : no policy and no cap, the delegation policy in the prompt, a cap of one task call per run in code, and both. The policy text is Anthropic's sample for Claude ; we measure it on the course model. Each arm runs three times, and we report the median.

In [7]:
import statistics
from langchain.agents.middleware import ToolCallLimitMiddleware

POLICY = ("Delegate to a subagent only for large tasks that are genuinely independent and parallelizable, "
          "such as a wide multi-file investigation. Do not delegate work you can finish yourself in a handful "
          "of tool calls, and do not use subagents to verify or double-check your own work. If one subagent "
          "can complete the task, use one rather than several, and keep spawn counts low.")
DESK = "You run the Veenhaven Bus service desk. Answer in one or two sentences. Do not write files."
specialists = [
    {"name": "parts", "description": "Looks up stock and depot location for a part number.",
     "system_prompt": "Use part_stock. Return stock and location.", "tools": [part_stock]},
    {"name": "manuals", "description": "Finds the first action and replacement condition for an error code.",
     "system_prompt": "Use fault_guidance. Return the first action and replacement condition.",
     "tools": [fault_guidance]},
]

def desk(policy, cap, direct, limit):
    middleware = [ToolCallLimitMiddleware(tool_name="task", run_limit=limit)] if cap else []
    return create_deep_agent(model=llm, tools=[part_stock, fault_guidance] if direct else [],
                             subagents=specialists, middleware=middleware,
                             system_prompt=DESK + (" " + POLICY if policy else ""))

def complete(text):
    lowered = text.lower()
    return ("6" in lowered or "six" in lowered) and "rietpoort" in lowered and "fan" in lowered

rows = []

def run_arm(arm, policy, cap, setting="desk tools", question=QUESTION, check=complete, limit=1):
    agent = desk(policy, cap, direct=setting == "desk tools", limit=limit)
    for run in range(1, 4):
        counter.reset()
        result = agent.invoke(ask(question), LIMIT)
        requested = [call for m in result["messages"] if m.type == "ai"
                     for call in m.tool_calls if call["name"] == "task"]
        blocked = [m for m in result["messages"]
                   if m.type == "tool" and m.name == "task" and m.status == "error"]
        rows.append(dict(setting=setting, arm=arm, run=run, requested=len(requested), ran=len(requested) - len(blocked),
                         direct=sum(1 for m in result["messages"] if m.type == "tool" and m.name != "task"),
                         calls=counter.calls, input_tokens=counter.input, output_tokens=counter.output,
                         complete=check(answer(result)), errors=counter.errors))
        print(json.dumps(rows[-1]))
    print("last answer :", answer(result))

def summary(setting):
    print(f"{'arm':12} {'requested':>9} {'ran':>4} {'direct':>6} {'calls':>5} {'input':>7} {'output':>6} complete")
    for arm in ["none", "policy", "cap", "policy+cap"]:
        group = [row for row in rows if row["setting"] == setting and row["arm"] == arm]
        med = {key: statistics.median(row[key] for row in group)
               for key in ["requested", "ran", "direct", "calls", "input_tokens", "output_tokens"]}
        print(f"{arm:12} {med['requested']:>9} {med['ran']:>4} {med['direct']:>6} {med['calls']:>5} "
              f"{med['input_tokens']:>7} {med['output_tokens']:>6} {sum(row['complete'] for row in group)}/3")

In [8]:
run_arm("none", policy=False, cap=False)

{"setting": "desk tools", "arm": "none", "run": 1, "requested": 0, "ran": 0, "direct": 2, "calls": 2, "input_tokens": 4403, "output_tokens": 104, "complete": true, "errors": 0}


{"setting": "desk tools", "arm": "none", "run": 2, "requested": 0, "ran": 0, "direct": 2, "calls": 2, "input_tokens": 4403, "output_tokens": 107, "complete": true, "errors": 0}


{"setting": "desk tools", "arm": "none", "run": 3, "requested": 0, "ran": 0, "direct": 2, "calls": 2, "input_tokens": 4403, "output_tokens": 105, "complete": true, "errors": 0}
last answer : There are 6 CE-CF-02 traction cooling fans in stock at Rietpoort. For F-CE-110, check the traction cooling fan first; replace it if airflow is below 80%.


In [9]:
run_arm("policy", policy=True, cap=False)

{"setting": "desk tools", "arm": "policy", "run": 1, "requested": 0, "ran": 0, "direct": 2, "calls": 2, "input_tokens": 4555, "output_tokens": 112, "complete": true, "errors": 0}


{"setting": "desk tools", "arm": "policy", "run": 2, "requested": 0, "ran": 0, "direct": 2, "calls": 2, "input_tokens": 4555, "output_tokens": 108, "complete": true, "errors": 0}


{"setting": "desk tools", "arm": "policy", "run": 3, "requested": 0, "ran": 0, "direct": 2, "calls": 2, "input_tokens": 4555, "output_tokens": 104, "complete": true, "errors": 0}
last answer : There are 6 CE-CF-02 cooling fans in stock at Rietpoort. For F-CE-110, first check the traction cooling fan; replace it if airflow is below 80%.


In [10]:
run_arm("cap", policy=False, cap=True)

{"setting": "desk tools", "arm": "cap", "run": 1, "requested": 0, "ran": 0, "direct": 2, "calls": 2, "input_tokens": 4403, "output_tokens": 104, "complete": true, "errors": 0}


{"setting": "desk tools", "arm": "cap", "run": 2, "requested": 0, "ran": 0, "direct": 2, "calls": 2, "input_tokens": 4403, "output_tokens": 104, "complete": true, "errors": 0}


{"setting": "desk tools", "arm": "cap", "run": 3, "requested": 0, "ran": 0, "direct": 2, "calls": 2, "input_tokens": 4403, "output_tokens": 104, "complete": true, "errors": 0}
last answer : There are 6 CE-CF-02 cooling fans in stock at Rietpoort. For F-CE-110, check the traction cooling fan first; replace it if airflow is below 80%.


In [11]:
run_arm("policy+cap", policy=True, cap=True)

{"setting": "desk tools", "arm": "policy+cap", "run": 1, "requested": 0, "ran": 0, "direct": 2, "calls": 2, "input_tokens": 4555, "output_tokens": 108, "complete": true, "errors": 0}


{"setting": "desk tools", "arm": "policy+cap", "run": 2, "requested": 0, "ran": 0, "direct": 2, "calls": 2, "input_tokens": 4555, "output_tokens": 104, "complete": true, "errors": 0}


{"setting": "desk tools", "arm": "policy+cap", "run": 3, "requested": 0, "ran": 0, "direct": 2, "calls": 2, "input_tokens": 4555, "output_tokens": 115, "complete": true, "errors": 0}
last answer : There are **6 CE-CF-02 cooling fans** in stock at **Rietpoort**. For F-CE-110, the manual says to **check the traction cooling fan** and replace it if airflow is **below 80%**.


The medians over three runs per arm are the result to read. `requested` counts `task` calls the model made, `ran` the ones the cap let through, and `direct` the bus tools the desk called itself.

In [12]:
summary("desk tools")

arm          requested  ran direct calls   input output complete
none                 0    0      2     2    4403    105 3/3
policy               0    0      2     2    4555    108 3/3
cap                  0    0      2     2    4403    104 3/3
policy+cap           0    0      2     2    4555    108 3/3


The cap counts every requested `task` call in one run. A blocked call returns an error message to the model, and the model continues with what it has. A policy in the prompt is advice the model may follow ; the cap holds whatever the model decides. When the desk has the bus tools itself, the course model may not delegate at all, and then neither control has anything to change.

## d · delegation-with-specialists-only

We take the bus tools away from the desk, so every lookup has to go through task, and we ask about four error codes and one part. The cap now allows two task calls per run. The same four arms run three times each.

In [13]:
WIDE = ("Buses report F-CE-110, F-CE-305, F-CD-101 and F-MM-100. Give the manual's first action for each code, "
        "and the stock and location of CE-CF-02. Be brief.")

def wide_complete(text):
    upper = text.upper()
    return all(code in upper for code in ["F-CE-110", "F-CE-305", "F-CD-101", "F-MM-100"]) and "RIETPOORT" in upper

run_arm("none", policy=False, cap=False, setting="specialists only", question=WIDE, check=wide_complete, limit=2)

{"setting": "specialists only", "arm": "none", "run": 1, "requested": 2, "ran": 2, "direct": 0, "calls": 6, "input_tokens": 11582, "output_tokens": 498, "complete": true, "errors": 0}


{"setting": "specialists only", "arm": "none", "run": 2, "requested": 5, "ran": 5, "direct": 0, "calls": 12, "input_tokens": 21795, "output_tokens": 585, "complete": true, "errors": 0}


{"setting": "specialists only", "arm": "none", "run": 3, "requested": 2, "ran": 2, "direct": 0, "calls": 6, "input_tokens": 11536, "output_tokens": 463, "complete": true, "errors": 0}
last answer : F-CE-110: Check traction cooling fan airflow; F-CE-305: inspect the door actuator; F-CD-101: replace the engine air filter; F-MM-100: run the ramp test in the workshop. CE-CF-02: **6 in stock**, **Rietpoort**.


In [14]:
run_arm("policy", policy=True, cap=False, setting="specialists only", question=WIDE, check=wide_complete, limit=2)

{"setting": "specialists only", "arm": "policy", "run": 1, "requested": 2, "ran": 2, "direct": 0, "calls": 6, "input_tokens": 11669, "output_tokens": 438, "complete": true, "errors": 0}


{"setting": "specialists only", "arm": "policy", "run": 2, "requested": 1, "ran": 1, "direct": 0, "calls": 6, "input_tokens": 11579, "output_tokens": 271, "complete": false, "errors": 0}


{"setting": "specialists only", "arm": "policy", "run": 3, "requested": 2, "ran": 2, "direct": 0, "calls": 6, "input_tokens": 11724, "output_tokens": 516, "complete": true, "errors": 0}
last answer : - **F-CE-110:** Check the traction cooling fan; replace if airflow is below 80%.
- **F-CE-305:** Inspect and replace the door actuator.
- **F-CD-101:** Replace the engine air filter; check intake pressure after the workshop test cycle.
- **F-MM-100:** Run the ramp test in the workshop; replace the motor if it stalls twice.
- **CE-CF-02:** 6 in stock at Rietpoort.


In [15]:
run_arm("cap", policy=False, cap=True, setting="specialists only", question=WIDE, check=wide_complete, limit=2)

{"setting": "specialists only", "arm": "cap", "run": 1, "requested": 2, "ran": 2, "direct": 0, "calls": 6, "input_tokens": 11541, "output_tokens": 461, "complete": true, "errors": 0}


{"setting": "specialists only", "arm": "cap", "run": 2, "requested": 2, "ran": 2, "direct": 0, "calls": 6, "input_tokens": 11557, "output_tokens": 463, "complete": true, "errors": 0}


{"setting": "specialists only", "arm": "cap", "run": 3, "requested": 4, "ran": 2, "direct": 0, "calls": 7, "input_tokens": 14050, "output_tokens": 529, "complete": false, "errors": 0}
last answer : F-CE-110: check the traction cooling fan; F-CE-305: inspect the door actuator; F-CD-101: replace the engine air filter; F-MM-100: run the ramp test in the workshop. I couldn’t retrieve the stock or location for CE-CF-02.


In [16]:
run_arm("policy+cap", policy=True, cap=True, setting="specialists only", question=WIDE, check=wide_complete, limit=2)

{"setting": "specialists only", "arm": "policy+cap", "run": 1, "requested": 2, "ran": 2, "direct": 0, "calls": 6, "input_tokens": 11633, "output_tokens": 394, "complete": true, "errors": 0}


{"setting": "specialists only", "arm": "policy+cap", "run": 2, "requested": 2, "ran": 2, "direct": 0, "calls": 6, "input_tokens": 11771, "output_tokens": 504, "complete": true, "errors": 0}


{"setting": "specialists only", "arm": "policy+cap", "run": 3, "requested": 1, "ran": 1, "direct": 0, "calls": 6, "input_tokens": 11597, "output_tokens": 289, "complete": false, "errors": 0}
last answer : I can’t verify the manual’s first actions for F-CE-110, F-CE-305, F-CD-101, or F-MM-100, or the stock and location of CE-CF-02, without access to the manuals and inventory records.


In [17]:
summary("specialists only")

arm          requested  ran direct calls   input output complete
none                 2    2      0     6   11582    498 3/3
policy               2    2      0     6   11669    438 2/3
cap                  2    2      0     6   11557    463 2/3
policy+cap           2    2      0     6   11633    394 2/3


`complete` checks that the answer names all four codes and the depot. It does not check each action against the manual ; we read the last answers above for that. A cap that blocks a lookup can leave the answer incomplete, which is the price of a hard limit.

## Things to try

Optional : we can change the bus task and compare the answer, calls and tokens.